# Lab 10 — Partitioned Files and Pruning

**Tujuan:** Memahami directory partition, partition pruning, dan beda dengan executor partitions.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 10")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
base = "/home/spark/outputs/lab10"
df = spark.range(0, 120_000, 1, 4)
data = df.withColumn("year", F.lit(2026))
data = data.withColumn("month", (F.col("id") % 12) + 1)
data = data.withColumn("amount", (F.col("id") % 5000) + 100)
data.write.mode("overwrite").partitionBy("month").parquet(base + "/sales")

## 1. Inspect generated layout

Driver dan workers berbagi `/home/spark/outputs`.

In [ ]:
from pathlib import Path
root = Path(base + "/sales")
print("Directories:", sorted(p.name for p in root.iterdir() if p.is_dir()))

## 2. Partition pruning

In [ ]:
sales = spark.read.parquet(base + "/sales")
january = sales.filter("month = 1")
all_months = sales.filter("amount > 1000")
print("January query:")
january.explain("formatted")
print("Amount-only query:")
all_months.explain("formatted")
print("January rows:", january.count())

**Tugas:** temukan `PartitionFilters` pada plan dan jelaskan mengapa folder `month=...` berbeda konsep dari `df.repartition(8)`.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.